# Tool catalog: camera, microphone, and screen

Run this notebook in a disposable JupyterLab project on a secure origin with a live Python kernel and selected ChatGPT connection. These are **real** camera, microphone, and screen operations, not synthetic device fixtures. Point the camera at a neutral wall, keep private windows outside the chosen screen area, and speak or make a brief sound for the microphone level. Grant browser permission only when its visible prompt appears. Each call returns a receipt quickly; wait for the media row to finish, then inspect it in the later cell. Recordings here run for at least five seconds. The final AI questions repeat bounded workflows through actual ChatGPT tool calls.


In [1]:
from nbinlineai.tools import (
    list_media_sources, start_camera, start_microphone, capture_camera, stop_source,
    start_recording, pause_recording, resume_recording, stop_recording,
    record_camera, record_microphone, read_audio_levels,
    setup_share, start_share, capture_screen, capture_tool, stop_share,
    operation_status, release_media, list_outputs, list_canvases, start_canvas,
)
from PIL import Image as PILImage
from IPython.display import display, HTML, Javascript

def public_media_result(value):
    if isinstance(value, dict):
        return {
            key: ('[withheld]' if key.lower().replace('_', '').replace('-', '')
                  in {'deviceid', 'groupid', 'sourceid', 'label'}
                  else public_media_result(item))
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [public_media_result(item) for item in value]
    if isinstance(value, bytes):
        return f'[{len(value)} media bytes withheld]'
    return value

def show_media_receipt(name, receipt):
    print(name, receipt.status, receipt.error)
    if receipt.status != 'completed':
        return
    result = receipt.result
    if isinstance(result, PILImage.Image):
        print('Captured image:', type(result).__name__, result.size)
    elif hasattr(result, 'data') and hasattr(result, 'mime_type'):
        print('Recorded clip:', type(result).__name__, result.mime_type, len(result.data), 'bytes', getattr(result, 'duration_seconds', None), 'seconds')
    elif isinstance(result, dict) and 'devices' in result:
        print('Available devices:', len(result['devices']), [item.get('kind') for item in result['devices']])
    elif isinstance(result, dict):
        print('Result:', public_media_result(result))
    else:
        print('Result:', result)

# Each media path can be run on its own before the shared cleanup.
camera_id = None
still = None
camera_clip = None
recording = None
microphone_clip = None
screenshot = None
alias_frame = None


## list_media_sources

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [2]:
devices = list_media_sources()

Real hardware source discovery completed; labels and identifiers withheld from public copy.


In [3]:
show_media_receipt('list_media_sources', devices)


Real hardware source discovery completed; labels and identifiers withheld from public copy.


## start_camera

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [2]:
camera = start_camera(audio=False)

In [3]:
show_media_receipt('start_camera', camera)


start_camera completed None
Result: {'source_id': '[private device detail]', 'kind': 'camera', 'audio': False, 'video': True, 'width': 640, 'height': 480}


## capture_camera

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [4]:
camera_id = camera.result['source_id'] if camera.status == 'completed' else None
still = capture_camera(camera_id, save_to=None) if camera_id else None


In [5]:
if still is not None:
    show_media_receipt('capture_camera', still)
else:
    print('Camera capture needs a completed camera source; continue with the microphone examples.')


capture_camera completed None
Captured image: PngImageFile (640, 480)


## start_microphone

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [4]:
microphone = start_microphone()

In [5]:
show_media_receipt('start_microphone', microphone)


start_microphone completed None
Result: {'source_id': '[opaque hardware reference]', 'kind': 'microphone', 'audio': True, 'video': False}


## read_audio_levels

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [6]:
microphone_id = microphone.result["source_id"]
levels = read_audio_levels(microphone_id)

In [7]:
show_media_receipt('read_audio_levels', levels)


read_audio_levels completed None
Result: {'rms': 0.006, 'peak': 0.008, 'window_ms': 250}


## start_recording

Record from the live microphone source for at least three active seconds before pausing. Resume for at least three more active seconds before stopping. The 60-second wall-clock ceiling continues while paused; the resulting real audio clip should have at least five playable seconds.


In [8]:
recording = start_recording(microphone_id, save_to=None, duration=60)


In [9]:
show_media_receipt('start_recording', recording)


start_recording running None


## pause_recording

Pause after at least three seconds of active microphone recording. Paused time produces no playable audio, but the 60-second wall-clock ceiling still applies.


In [10]:
paused = pause_recording(recording.operation_id)

In [11]:
show_media_receipt('pause_recording', paused)


pause_recording completed None
Result: {'target_operation_id': 'AcMIg9sF3PEWvJccJQTQ2UJiXg4nJCGR', 'paused': True, 'tracks_remain_live': True}


## resume_recording

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [12]:
resumed = resume_recording(recording.operation_id)

In [13]:
show_media_receipt('resume_recording', resumed)


resume_recording completed None
Result: {'target_operation_id': 'AcMIg9sF3PEWvJccJQTQ2UJiXg4nJCGR', 'resumed': True}


## stop_recording

Wait at least three active seconds after resuming before stopping; combined active recording should be at least six seconds. Inspect the clip summary without printing its raw bytes.


In [14]:
stopped_recording = stop_recording(recording.operation_id)

In [15]:
show_media_receipt('stop_recording control', stopped_recording)
show_media_receipt('original recording after stop', recording)
if recording.status != 'completed':
    print('Wait for the original recording receipt to complete, then rerun this inspection cell.')


stop_recording control completed None
Result: {'target_operation_id': 'AcMIg9sF3PEWvJccJQTQ2UJiXg4nJCGR', 'target_status': 'completed'}
original recording after stop completed None
Recorded clip: MediaClip audio/webm;codecs=opus 332785 bytes 24.629099999904632 seconds


## stop_source

Stop the microphone source after the recording finishes. Inspect this control receipt on a later kernel turn; it does not remove the completed audio clip.


In [16]:
stopped_microphone = stop_source(microphone_id)


In [17]:
show_media_receipt('stop_source', stopped_microphone)


stop_source completed None
Result: {'source_id': '[opaque hardware reference]', 'stopped': True}


## record_camera

Record the neutral wall for five seconds. Allow camera permission in the browser if prompted, then inspect the actual clip type, byte count, and duration.


In [6]:
camera_clip = record_camera(save_to=None, duration=5, audio=False)

In [7]:
show_media_receipt('record_camera', camera_clip)


record_camera completed None
Recorded clip: MediaClip video/webm;codecs=vp9 1479368 bytes 5 seconds


## record_microphone

Request seven seconds of real microphone audio so the decoded clip comfortably exceeds five seconds. A quiet voice or brief sound helps make the level reading meaningful. Grant permission in the browser if prompted.


In [18]:
microphone_clip = record_microphone(save_to=None, duration=7)

In [19]:
show_media_receipt('record_microphone', microphone_clip)


record_microphone completed None
Recorded clip: MediaClip audio/webm;codecs=opus 112313 bytes 7 seconds


## setup_share

Check the browser's screen-sharing controls before sharing. Keep private windows out of view.


In [2]:
sharing_controls = setup_share()

In [3]:
show_media_receipt('setup_share', sharing_controls)


setup_share completed None
Result: {'share_controls': True, 'screen_available': True}


## start_share

Click the browser's visible Share chooser and select only a neutral tab or area of this disposable project. Screen capture never starts silently.


In [4]:
sharing = start_share(audio=False)

In [5]:
show_media_receipt('start_share', sharing)


start_share completed None
Result: {'source_id': '[private device detail]', 'kind': 'screen', 'audio': False, 'video': True, 'width': 1500, 'height': 786, 'display_surface': 'browser'}


## capture_screen

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [6]:
screen_id = sharing.result["source_id"]
screenshot = capture_screen(source_id=screen_id, save_to=None)

In [7]:
show_media_receipt('capture_screen', screenshot)


capture_screen completed None
Captured image: PngImageFile (1280, 670)


## capture_tool

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [8]:
alias_frame = capture_tool(source_id=screen_id, save_to=None)

In [9]:
show_media_receipt('capture_tool', alias_frame)


capture_tool completed None
Captured image: PngImageFile (1280, 670)


## stop_share

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [10]:
stopped_share = stop_share(screen_id)

In [11]:
show_media_receipt('stop_share', stopped_share)


stop_share completed None
Result: {'source_id': '[private device detail]', 'stopped': True}


The microphone and screen sources are stopped above. The next cleanup stops the wall camera if it was available and releases only managed media IDs created here. Delivered Pillow images and clips remain ordinary Python objects until discarded.


In [8]:
camera_cleanup = stop_source(camera_id) if camera_id else None
released_media = {}
seen_media_ids = set()
for label, receipt in [
    ('wall still', still), ('microphone recording', recording),
    ('camera clip', camera_clip), ('microphone clip', microphone_clip),
    ('screen still', screenshot), ('screen alias', alias_frame),
]:
    descriptor = receipt.media if receipt is not None and receipt.status == 'completed' else None
    media_id = descriptor.get('media_id') if isinstance(descriptor, dict) else None
    if media_id and media_id not in seen_media_ids:
        seen_media_ids.add(media_id)
        released_media[label] = release_media(media_id)
print('Requested release for', len(released_media), 'managed captures.')


Requested release for 2 managed captures.


In [9]:
if camera_cleanup is not None:
    show_media_receipt('camera cleanup', camera_cleanup)
for label, receipt in released_media.items():
    show_media_receipt(f'release {label}', receipt)


camera cleanup completed None
Result: {'source_id': '[private device detail]', 'stopped': True}
release wall still completed None
Result: {'released': True}
release camera clip completed None
Result: {'released': True}


## Ask ChatGPT to use the live devices

Run these short questions one at a time with the selected ChatGPT connection. A browser action first returns an operation ID and a running or waiting state; after your permission or recording interval, the separate status question checks the terminal result before another action uses its source ID. Keep each saved answer so the next question can use the exact ID. If camera access is unavailable, continue with the independent microphone recording and screen-sharing sections. Grant permissions and choose the neutral screen area yourself.


Call &`list_media_sources` with `kind="all"` and `limit=10` to list devices passively. Report only the immediate operation ID and current state. Do not request camera permission, report device counts yet, or start a camera.


Operation ID: `[withheld; see linked status]` — state: `running`.

**Observed live tool call for the answer above** (ChatGPT subscription; private IDs and media bytes withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `list_media_sources` | receipt accepted | `kind, limit` | Observed tool result; operation state: running. |


Use &`operation_status` on the preceding `list_media_sources` operation ID after its media row finishes. If completed, report the camera and microphone counts on the returned page, whether more pages remain, and any omitted count, without labels or device IDs. If it is still active, stop and ask me to retry this status question later.


Completed. Returned page: 1 camera and 5 microphones; no more pages, 0 omitted. Device labels and IDs are withheld.

**Observed live tool call for the answer above** (ChatGPT subscription; private IDs and media bytes withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. |


Call &`start_camera` with `audio=False` for the scene I approve. Report only its immediate operation ID and current state. Ask me to grant the visible camera permission; do not claim an active source yet.


Operation ID: `xAo5iRSF5O3KdxtNZaDRYyX0qlM1UdUf`. State: `waiting_for_user`. Please grant the visible camera permission.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `start_camera` | receipt accepted | `audio` | Observed tool result; operation state: waiting_for_user. |


After I grant camera permission, use &`operation_status` on the preceding `start_camera` operation ID. Report its verified source ID only if the operation completed. If it remains waiting or running, stop and retry this status question later before capturing.


Verified source ID: `[opaque source ID]`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. 640 × 480 pixels. |


Using only the completed camera source ID from the preceding status answer, call &`capture_camera` with `save_to=None`. Report its immediate operation ID/state; do not claim pixels or dimensions yet.


Operation ID: `0Zva1rRg3rFtMNrtIKskwQaCn9Nk1E5G`. State: `running`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `capture_camera` | receipt accepted | `save_to`, `source_id` | Observed tool result; operation state: running. |


Use &`operation_status` on the prior `capture_camera` operation ID after its media row completes. Report actual image dimensions, MIME type, and managed media ID if present, without pixels or personal device details. If still running, stop and ask me to retry later.


Capture completed. Image: 640 × 480, MIME type `image/png`, managed media ID `SpZowYry8st8hhalvGCWoquY1_sGw9XK`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. Managed media ID returned; bytes omitted. 640 × 480 pixels. |


Use &`stop_source` on the exact camera source ID confirmed above. Report the immediate control operation ID/state, and do not claim a different source was stopped.


Stop operation ID: `4sUf-w1uYHWXUU-bn0lX2jyj4VEsb0V2`. State: `running`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `stop_source` | receipt accepted | `source_id` | Observed tool result; operation state: running. |


Use &`operation_status` on the preceding camera `stop_source` operation ID after its media row finishes. Confirm the exact source is stopped before continuing with the separate microphone recording. If still active, stop here and retry this status question later.


Call &`start_microphone` to request a real microphone source for this recording path. I will grant the browser permission when prompted. Report its immediate operation ID/state; do not begin recording until the source is confirmed by `operation_status`.


Operation ID: `zpIK2UJwSLzVjHaf8j5D0eQjQX2gWqK4` — waiting for you. Please grant microphone permission when the browser prompt appears.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `start_microphone` | receipt accepted | none | Observed tool result; operation state: waiting_for_user. |


Use &`operation_status` on the preceding recording-workflow `start_microphone` operation ID after I grant permission. Report its actual state and source ID only when completed; otherwise stop and ask me to retry later.


Completed. Microphone source ID: `[opaque source ID]`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. |


Use &`stop_source` on the microphone source ID confirmed above. Report the immediate stop operation ID/state; a later status question confirms terminal cleanup.


Stop operation ID: `_XYZYeZlqT9ISeQVkqIAiM4GXQ8BlBXa` — running.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `stop_source` | receipt accepted | `source_id` | Observed tool result; operation state: running. |


Use &`operation_status` on the preceding microphone `stop_source` operation ID. Confirm the source reached stopped state.


The stop operation completed: the recording microphone source is stopped. You can start another microphone for the level reading.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. Source stopped. |


### Recording controls on a generated canvas

The following independent browser trial used a 64 × 32 animated canvas with no microphone, camera, or audio track. It demonstrates real browser `MediaRecorder` pause, resume, and stop operations. The first recording reached its 300-second wall limit shortly after resume; a second recording demonstrates an explicit stop. In the private trial, the exact source and operation IDs were pasted into the questions; saved IDs are withheld here. When rerunning, use the current IDs printed or returned by your own cells and answers.


In [2]:
display(HTML('<canvas id="nbinlineai-recorder-canvas" width="64" height="32">Generated animated canvas</canvas>'))
display(Javascript("""(() => { const canvas = document.getElementById('nbinlineai-recorder-canvas'); if (!canvas) return; const ctx = canvas.getContext('2d'); let frame = 0; const timer = setInterval(() => { if (!canvas.isConnected) { clearInterval(timer); return; } ctx.fillStyle = frame++ % 2 ? '#2542a3' : '#d4402f'; ctx.fillRect(0, 0, 64, 32); }, 100); })();"""))


<IPython.core.display.Javascript object>

In [3]:
canvas_outputs = list_outputs('canvas-recorder-output')


In [4]:
canvas_output_ref = canvas_outputs.result['outputs'][0]
canvas_page = list_canvases(canvas_output_ref['cell_id'], canvas_output_ref['output_id'], canvas_output_ref['revision'])


In [5]:
canvas_ref = canvas_page.result['canvases'][0]
canvas_source = start_canvas(canvas_ref, frame_rate=12)


In [6]:
canvas_source_id = canvas_source.result['source_id']
print('Generated canvas source active:', canvas_source.status, canvas_source.result['kind'], canvas_source.result['width'], canvas_source.result['height'])

print('Canvas source ID:', canvas_source_id)

Generated canvas source active: completed canvas 64 32
Canvas source ID: [withheld in saved example; rerun to see current ID]


Use &`start_recording` once on the generated canvas source ID from the executed cell above, with `save_to="auto"` and `duration=300`. Paste the exact current source ID into this question before running. Report the immediate operation state. This source has no audio.


The first generated-canvas recording was accepted and entered `running`; its operation ID is withheld.

**Observed live tool call for the answer above** (ChatGPT subscription; private IDs and media bytes withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `start_recording` | receipt accepted | `duration, save_to, source_id` | Observed tool result; operation state: running. |


After at least three seconds of active recording, call &`pause_recording` exactly once with the first generated-canvas recording operation ID returned above. Paste that exact current ID before running. Report the immediate control receipt.


The pause control was accepted; its own operation was initially `running`. The control ID is withheld.

**Observed live tool call for the answer above** (ChatGPT subscription; private IDs and media bytes withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `pause_recording` | receipt accepted | `operation_id` | Observed tool result; operation state: running. |


Use &`operation_status` exactly once on the first generated-canvas recording operation ID, not the pause-control ID. Confirm its current state before resuming.


The original generated-canvas recording is `paused`.

**Observed live tool call for the answer above** (ChatGPT subscription; private IDs and media bytes withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: paused. |


Call &`resume_recording` exactly once with the first generated-canvas recording operation ID whose paused state was confirmed above. Paste that exact current ID before running. Report the immediate control receipt.


The resume control was accepted; its own operation was initially `running`. The control ID is withheld.

**Observed live tool call for the answer above** (ChatGPT subscription; private IDs and media bytes withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `resume_recording` | receipt accepted | `operation_id` | Observed tool result; operation state: running. |


**Observed browser state:** After the resume control, the operator saw the original recording’s Media row change from `paused` to `running`. It later completed at the 300-second wall limit before a separate AI status check. This is a UI observation, not an AI status result.


Call &`start_recording` exactly once on the same active generated-canvas source ID, with `save_to="auto"` and `duration=300`. Paste the exact current source ID before running. This second recording is for the explicit stop demonstration; it has no audio.


The second generated-canvas recording was accepted and entered `running`; its operation ID is withheld.

**Observed live tool call for the answer above** (ChatGPT subscription; private IDs and media bytes withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `start_recording` | receipt accepted | `duration, save_to, source_id` | Observed tool result; operation state: running. |


After several active seconds on the second generated-canvas recording, call &`stop_recording` exactly once with that second recording operation ID, not the first one. Paste the exact current ID before running. Report the immediate stop-control receipt.


The stop control was accepted; its own operation was initially `running`. The control ID is withheld.

**Observed live tool call for the answer above** (ChatGPT subscription; private IDs and media bytes withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `stop_recording` | receipt accepted | `operation_id` | Observed tool result; operation state: running. |


Use &`operation_status` exactly once on the second generated-canvas recording operation ID after its media row completes. Report terminal state, duration, MIME type, byte size, and whether a managed media ID was returned. Do not include a local path or media bytes.


Completed. Duration: 54.5537 seconds; MIME type: `video/webm;codecs=vp9`; size: 35,728 bytes. A managed media ID was returned; the ID and encoded video are withheld.

**Observed live tool call for the answer above** (ChatGPT subscription; private IDs and media bytes withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. 54.5537 seconds, 35,728 bytes; video bytes omitted. |


In [7]:
canvas_stop = stop_source(canvas_source_id)
print("Generated canvas stop requested:", canvas_stop.status)


Generated canvas stop requested: running


The generated canvas source was stopped after the recording test. The browser-managed video was kept only in the disposable test environment; this notebook saves its receipt metadata, not video bytes or a local path.


Use &`start_microphone` after I grant real microphone permission. Report the immediate operation ID/state, without claiming a source ID until it completes.


Operation ID: `zesyj2AZf2CyuJyL84UdrGK4WzUC-sek` — waiting for you.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `start_microphone` | receipt accepted | none | Observed tool result; operation state: waiting_for_user. |


Use &`operation_status` on the previous `start_microphone` operation ID after permission. Report the actual state and source ID only if completed; otherwise stop and retry later.


Completed. Microphone source ID: `[opaque source ID]`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. |


While I make a brief sound, use &`read_audio_levels` on the confirmed microphone source ID. Report the immediate operation ID/state, not an invented level.


Operation ID: `y8HhO-kJtIv3kbgj8_Jd7v7BYItbCfnv` — running.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `read_audio_levels` | receipt accepted | `source_id` | Observed tool result; operation state: running. |


Use &`operation_status` on the preceding `read_audio_levels` operation ID after it completes. Report the actual RMS level and units if present; if still running, stop and retry later.


Completed. RMS level: **0.006** (units not provided).

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. |


Use &`stop_source` on the exact microphone source ID from this workflow. Report its immediate control operation ID/state.


Control operation ID: `dnYi1LYHHyXgGfIcaYsIL8qQ1KX9b02U` — running.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `stop_source` | receipt accepted | `source_id` | Observed tool result; operation state: running. |


Use &`operation_status` on the preceding microphone `stop_source` operation ID. Confirm terminal stopped state before the later one-shot recording; if not complete, stop and retry later.


The microphone stop completed: source `[opaque source ID]` is stopped. The later one-shot recording can proceed.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. Source stopped. |


Use &`record_camera` with `audio=False`, `duration=5`, and `save_to=None` for the scene I approve. Report only the immediate operation ID/state, then wait for permission and the five-second recording.


Operation ID: `Bw6jS-4VL-m950YT9cDvDCzuTFzeASKx`. State: `waiting_for_user`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `record_camera` | receipt accepted | `audio`, `duration`, `save_to` | Observed tool result; operation state: waiting_for_user. |


After the five-second wall-camera recording completes, use &`operation_status` on the preceding `record_camera` operation ID. Report actual duration, MIME type, byte count, and managed media ID if available; never include encoded bytes. If still running, stop and retry later.


Recording completed. Duration: 5 seconds; MIME type: `video/webm;codecs=vp9`; size: 1,196,251 bytes; managed media ID: `8bfgw-9Z-6r81V_fYxmbQCOj0Uw9suQl`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. Managed media ID returned; bytes omitted. |


Use &`record_microphone` with `duration=7` and `save_to=None` after I grant real microphone permission. Report only its immediate operation ID/state, then wait for the recording.


Operation ID: `arW3gB277uPHqE1Id5DFNeBW2Nu7KIqa` — waiting for you.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `record_microphone` | receipt accepted | `duration`, `save_to` | Observed tool result; operation state: waiting_for_user. |


After the seven-second microphone recording completes, use &`operation_status` on the preceding `record_microphone` operation ID. Report actual duration, MIME type, byte count, and managed media ID if present; never include audio bytes. If still running, stop and retry later.


Completed. Duration: 7 seconds; MIME type: `audio/webm;codecs=opus`; size: 112,313 bytes; managed media ID: `N3WOus7A34BE43pUShkxa2TFyeKuG4GB`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. Managed media ID returned; bytes omitted. |


Use &`setup_share` to check actual screen-sharing support. Report its immediate operation ID/state and wait for the terminal result before choosing a screen; do not share anything silently.


Operation ID: `keUfnuxKy8x_pD9KtxKLpf5FeG43wCZb` — running.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `setup_share` | receipt accepted | none | Observed tool result; operation state: running. |


Use &`operation_status` on the `setup_share` operation ID after it completes. Report the actual support state. If unsupported, stop the screen workflow and explain that result.


Screen sharing is supported: Share controls and screen capture are available. Nothing has been shared yet.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. |


Use &`start_share` with `audio=False` to open the visible browser Share dialog. I will choose a neutral tab there. Report the immediate operation ID/state; do not claim a source ID until a later `operation_status` confirms completion.


Operation ID: `fpG-czbhwQn2-ascIjqeAyymXjYGm5Aj` — waiting for you to choose a tab in the visible Share dialog.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `start_share` | receipt accepted | `audio` | Observed tool result; operation state: waiting_for_user. |


After I choose the tab, use &`operation_status` on the `start_share` operation ID. Report the actual state and source ID only if completed; otherwise stop and retry later.


Completed. Screen source ID: `[opaque source ID]`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. 2880 × 1800 pixels. |


Use &`capture_screen` on the exact confirmed share source ID with `save_to=None`. Report its immediate operation ID/state, not image dimensions yet.


Operation ID: `C724Z0uZOOLgE5hsFjikJaV6XLr-44ck` — running.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `capture_screen` | receipt accepted | `save_to`, `source_id` | Observed tool result; operation state: running. |


Use &`operation_status` on the `capture_screen` operation ID after completion. Report actual image dimensions and managed media ID if present, not pixels or private screen content.


Completed. Image dimensions: 1280 × 800 pixels. Managed media ID: `NNl8ULCC7dZ0APXSq3CsBtgB8HfZ1-5o`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. Managed media ID returned; bytes omitted. 1280 × 800 pixels. |


Use &`capture_tool` with `source_id` from the immediately preceding completed AI `operation_status` answer for `start_share` (the screen source ID it reported), and `save_to=None`. Do not use the Python example's `screen_id` or any earlier source ID. Report only the immediate operation ID/state.


Operation ID: `m3DlUkt7My3nzwpHQZeumlgAI2znLvaF` — running.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `capture_tool` | receipt accepted | `save_to`, `source_id` | Observed tool result; operation state: running. |


Use &`operation_status` on the `capture_tool` operation ID after completion. Report actual dimensions and managed media ID if present; do not include pixels.


Completed. Dimensions: 1280 × 800 pixels. Managed media ID: `fwGEQt-x8vB2rY0ygYBZt-zbYmrb51xP`.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. Managed media ID returned; bytes omitted. 1280 × 800 pixels. |


Use &`stop_share` on the exact confirmed share source ID. Report its immediate stop operation ID/state.


Stop operation ID: `BDhkpqU3VGo2WIs-CtUu6FzLYV1sJXiY` — running.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `stop_share` | receipt accepted | `source_id` | Observed tool result; operation state: running. |


Use &`operation_status` on the preceding `stop_share` operation ID. Report its actual terminal state and confirm the exact share source stopped before cleanup.


Completed: share source `[opaque source ID]` is stopped and ready for cleanup.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | completed | `operation_id` | Observed tool result; operation state: completed. Source stopped. |


In one parallel tool group, call &`release_media` once for each of four distinct managed media references from completed earlier AI capture results: the microphone clip, an earlier 60-second microphone recording, the screen still, and the `capture_tool` still. Use the exact IDs from those results in your own live session. The earlier recording is separate from the generated-canvas recorder trial above. Report each release result; do not include media bytes.


- Microphone clip: release failed — media reference expired; no operation ID.
- 60-second recording: release accepted (`released: true`); operation ID `_7r9g6Ng2zMBbJfTEBBfBHHPZdR-x6ve`. No state was reported.
- Screen still: release failed — media reference expired; no operation ID.
- `capture_tool` still: release failed — media reference expired; no operation ID.

Releasing managed browser bytes does not erase saved notebook displays.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `release_media` | failed | `media_id` | Reported error; no successful media result. |
| `release_media` | completed | `media_id` | Observed tool result; operation state: not logged. |
| `release_media` | failed | `media_id` | Reported error; no successful media result. |
| `release_media` | failed | `media_id` | Reported error; no successful media result. |


Use &`operation_status` on the one release operation ID reported as accepted above. Report the actual status or error. The other three media references expired and produced no release operation IDs; do not invent them.


The 60-second recording’s release was accepted (`released: true`), but its status lookup returned `stale_target`, so terminal completion cannot be confirmed. The microphone clip, screen still, and `capture_tool` still failed to release because their media references had expired.

**Observed live tool calls for the answer above** (ChatGPT subscription; private values withheld).

| Tool | Result state | Submitted fields | Observed result |
| --- | --- | --- | --- |
| `operation_status` | failed | `operation_id` | Reported error; no successful media result. |


### What these live AI runs established

On 2026-09-29, these AI questions used the connected ChatGPT subscription, `gpt-6-sol`, and Compact mode in isolated JupyterLab sessions. The user granted browser microphone and camera access and selected the screen to share. Camera and screen images, camera video bytes, and microphone audio bytes are omitted from this public notebook; saved answers retain receipt metadata, and the tables record the actual bounded tool calls. The camera scene was approved by the user after the initial prompt; its still was a 640 × 480 PNG and its five-second clip was VP9/WebM.

A separate silent browser run used a generated 64 × 32 canvas. The first recording was confirmed `paused` by an AI status call; after the guarded `resume_recording` call, the operator saw its Media row change from `paused` to `running`, then it reached the 300-second wall limit before another AI status call. A second generated-canvas recording was explicitly stopped and confirmed complete: 54.5537 seconds, `video/webm;codecs=vp9`, 35,728 bytes. The generated video is omitted. This UI observation is labeled separately from AI tool results.

An earlier AI microphone recording reached its 60-second memory limit before its attempted pause could be confirmed; the direct Python microphone pause/resume/stop examples above retain their own real saved execution. Three older managed media references expired before cleanup. One `release_media` call returned `released: true`, but its later status lookup returned `stale_target`, so this notebook does not claim a confirmed terminal release status.
